In [1]:
%pip install torch matplotlib

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pyloudnorm

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install torchaudio

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import re
from features import *
import librosa


In [5]:
folder_path = os.path.expanduser("~/Downloads/mod_discovery-main/data/wavetables/ableton/")
pt_files = [f for f in os.listdir(folder_path) if f.endswith(".pt")]

sample_rate = 44100

loudness_metric = Loudness(sample_rate)
centroid_metric = SpectralCentroid(sample_rate, window="flat_top", compress=True, floor=1e-4, scaling="kazazis")
flatness_metric = SpectralFlatness()




In [20]:
output_dir = "OutputEDA_Features"
os.makedirs(output_dir, exist_ok=True)
for file in pt_files:
    
    wt = torch.load(os.path.join(folder_path, file))
    
    # --- Compute features ---
    loudness = loudness_metric(wt)
    centroid = centroid_metric(wt)
    flatness = flatness_metric(wt)
    
    # --- Compute range ---
    loud_range = (torch.max(loudness) - torch.min(loudness)).item()
    cent_range = (torch.max(centroid) - torch.min(centroid)).item()
    flat_range = (torch.max(flatness) - torch.min(flatness)).item()
    
    # --- Compute entropy ---
    def compute_entropy(x, bins=32):
        x = x.detach().cpu().numpy()
        hist, _ = np.histogram(x, bins=bins, density=True)
        hist += 1e-12
        prob = hist / np.sum(hist)
        return -np.sum(prob * np.log2(prob))
    
    loud_entropy = compute_entropy(loudness)
    cent_entropy = compute_entropy(centroid)
    flat_entropy = compute_entropy(flatness)
    
    # --- Plot raw feature trajectories ---
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,3,1)
    plt.plot(loudness.cpu().numpy())
    plt.title("Loudness")
    plt.xlabel("Frame")
    plt.ylabel("Loudness (in dBFS)")
    
    plt.subplot(1,3,2)
    plt.plot(centroid.cpu().numpy())
    plt.title("Spectral Centroid")
    plt.xlabel("Frame")
    plt.ylabel("Centroid (FFT bin index)")
    
    plt.subplot(1,3,3)
    plt.plot(flatness.cpu().numpy())
    plt.title("Spectral Flatness")
    plt.xlabel("Frame")
    plt.ylabel("Flatness (in dBFS)")
    
    plt.suptitle(f"{file}")
    plt.tight_layout()
    #plt.show()
    
    # --- Print summary ---
    print(f"\n{file} Statistics:")
    print(f"Loudness  → Range: {loud_range:.4f}, Entropy: {loud_entropy:.4f}")
    print(f"Centroid  → Range: {cent_range:.4f}, Entropy: {cent_entropy:.4f}")
    print(f"Flatness  → Range: {flat_range:.4f}, Entropy: {flat_entropy:.4f}")
    print("-" * 50)
    save_path = os.path.join(output_dir, f"{file.replace('.pt','')}_features.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


complex__menace__118_1024.pt Statistics:
Loudness  → Range: 11.5720, Entropy: 4.2048
Centroid  → Range: 3.8260, Entropy: 3.9616
Flatness  → Range: 48.9859, Entropy: 4.2892
--------------------------------------------------

harmonics__squarrely__174_1024.pt Statistics:
Loudness  → Range: 2.7362, Entropy: 4.5096
Centroid  → Range: 0.6734, Entropy: 4.3063
Flatness  → Range: 17.9005, Entropy: 4.6720
--------------------------------------------------

instrument__beauty_vox__130_1024.pt Statistics:
Loudness  → Range: 19.6519, Entropy: 4.4323
Centroid  → Range: 2.3472, Entropy: 4.4537
Flatness  → Range: 36.7283, Entropy: 4.6867
--------------------------------------------------

formant__ahohooh__3_1024.pt Statistics:
Loudness  → Range: 4.5049, Entropy: 1.5850
Centroid  → Range: 1.7852, Entropy: 1.5850
Flatness  → Range: 31.2527, Entropy: 1.5850
--------------------------------------------------

retro__sweep_pulse__184_1024.pt Statistics:
Loudness  → Range: 1.9673, Entropy: 4.8568
Centroi

In [15]:
def shannon_entropy(x, bins=32):
    x = x.detach().cpu().numpy()
    hist, _ = np.histogram(x, bins=bins, density=True)
    hist += 1e-12
    prob = hist / np.sum(hist)
    return -np.sum(prob * np.log2(prob))

def spectral_entropy(x):
    x = x.detach().cpu().numpy()
    fft = np.abs(np.fft.rfft(x))
    fft += 1e-12
    prob = fft / np.sum(fft)
    return -np.sum(prob * np.log2(prob))

def total_variation(x):
    return torch.sum(torch.abs(torch.diff(x))).item()

def turning_points(x):
    x = x.detach().cpu().numpy()
    count = 0
    for i in range(1, len(x)-1):
        if (x[i] > x[i-1] and x[i] > x[i+1]) or \
           (x[i] < x[i-1] and x[i] < x[i+1]):
            count += 1
    return count

In [16]:
import pandas as pd

rows = []

for file in pt_files:
    
    wt = torch.load(os.path.join(folder_path, file))
    
    loudness = loudness_metric(wt)
    centroid = centroid_metric(wt)
    flatness = flatness_metric(wt)
    
    def curve_stats(curve):
        return {
            "range": (torch.max(curve) - torch.min(curve)).item(),
            "entropy": shannon_entropy(curve),
            "spectral_entropy": spectral_entropy(curve),
            "max": torch.max(curve).item(),
            "min": torch.min(curve).item(),
            "total_variation": total_variation(curve),
            "turning_points": turning_points(curve)
        }
    
    loud_stats = curve_stats(loudness)
    cent_stats = curve_stats(centroid)
    flat_stats = curve_stats(flatness)
    
    row = {
        "wavetable_name": file,
        
        # Loudness
        "loud_range": loud_stats["range"],
        "loud_entropy": loud_stats["entropy"],
        "loud_spectral_entropy": loud_stats["spectral_entropy"],
        "loud_max": loud_stats["max"],
        "loud_min": loud_stats["min"],
        "loud_total_variation": loud_stats["total_variation"],
        "loud_turning_points": loud_stats["turning_points"],
        
        # Centroid
        "cent_range": cent_stats["range"],
        "cent_entropy": cent_stats["entropy"],
        "cent_spectral_entropy": cent_stats["spectral_entropy"],
        "cent_max": cent_stats["max"],
        "cent_min": cent_stats["min"],
        "cent_total_variation": cent_stats["total_variation"],
        "cent_turning_points": cent_stats["turning_points"],
        
        # Flatness
        "flat_range": flat_stats["range"],
        "flat_entropy": flat_stats["entropy"],
        "flat_spectral_entropy": flat_stats["spectral_entropy"],
        "flat_max": flat_stats["max"],
        "flat_min": flat_stats["min"],
        "flat_total_variation": flat_stats["total_variation"],
        "flat_turning_points": flat_stats["turning_points"],
    }
    
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv("OutputEDA_Features/wavetable_descriptors.csv", index=False)

print("Descriptor table saved.")

Descriptor table saved.


In [17]:
import json
json_path = os.path.expanduser("~/Downloads/wavetables_raw_vital.json")

In [18]:
with open(json_path, "r") as f:
    data = json.load(f)
wavetables = {}

for i, item in enumerate(data):
    
    # Extract frames
    frames = item["wavetables"]   # THIS is the key
    
    # Convert safely to numeric array
    wt_np = np.array(frames, dtype=np.float32)
    
    # Convert to torch tensor
    wt = torch.tensor(wt_np, dtype=torch.float32)
    
    # Give it a name
    name = f"vital_wavetable_{i}"
    
    wavetables[name] = wt

print(f"Loaded {len(wavetables)} wavetables.")
print("Example shape:", list(wavetables.values())[0].shape)


Loaded 228 wavetables.
Example shape: torch.Size([2, 2048])


In [19]:
output_dir = "OutputEDA_Features_Vital"
os.makedirs(output_dir, exist_ok=True)

for name, wt in wavetables.items():
    
    # --- Compute features ---
    loudness = loudness_metric(wt)
    centroid = centroid_metric(wt)
    flatness = flatness_metric(wt)
    
    # --- Compute range ---
    loud_range = (torch.max(loudness) - torch.min(loudness)).item()
    cent_range = (torch.max(centroid) - torch.min(centroid)).item()
    flat_range = (torch.max(flatness) - torch.min(flatness)).item()
    
    # --- Compute entropy ---
    loud_entropy = compute_entropy(loudness)
    cent_entropy = compute_entropy(centroid)
    flat_entropy = compute_entropy(flatness)
    
    # --- Plot ---
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,3,1)
    plt.plot(loudness.cpu().numpy(), marker="o")
    plt.title("Loudness")
    plt.xlabel("Frame")
    
    plt.subplot(1,3,2)
    plt.plot(centroid.cpu().numpy(), marker="o")
    plt.title("Spectral Centroid")
    plt.xlabel("Frame")
    
    plt.subplot(1,3,3)
    plt.plot(flatness.cpu().numpy(), marker="o")
    plt.title("Spectral Flatness")
    plt.xlabel("Frame")
    
    plt.suptitle(f"{name}")
    plt.tight_layout()
    
    # Save instead of show
    save_path = os.path.join(output_dir, f"{name}_features.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    
    # --- Print statistics ---
    print(f"\n{name} Statistics:")
    print(f"Loudness  → Range: {loud_range:.4f}, Entropy: {loud_entropy:.4f}")
    print(f"Centroid  → Range: {cent_range:.4f}, Entropy: {cent_entropy:.4f}")
    print(f"Flatness  → Range: {flat_range:.4f}, Entropy: {flat_entropy:.4f}")
    print("-" * 50)



vital_wavetable_0 Statistics:
Loudness  → Range: 0.8701, Entropy: 1.0000
Centroid  → Range: 0.0455, Entropy: 1.0000
Flatness  → Range: 9.1726, Entropy: 1.0000
--------------------------------------------------

vital_wavetable_1 Statistics:
Loudness  → Range: 0.0000, Entropy: 0.0000
Centroid  → Range: 0.0000, Entropy: 0.0000
Flatness  → Range: 0.0000, Entropy: 0.0000
--------------------------------------------------

vital_wavetable_2 Statistics:
Loudness  → Range: 0.0069, Entropy: 1.0000
Centroid  → Range: 0.0339, Entropy: 1.0000
Flatness  → Range: 0.0356, Entropy: 1.0000
--------------------------------------------------

vital_wavetable_3 Statistics:
Loudness  → Range: 0.0000, Entropy: 0.0000
Centroid  → Range: 0.0000, Entropy: 0.0000
Flatness  → Range: 0.0000, Entropy: 0.0000
--------------------------------------------------

vital_wavetable_4 Statistics:
Loudness  → Range: 0.0000, Entropy: 0.0000
Centroid  → Range: 0.0000, Entropy: 0.0000
Flatness  → Range: 0.0000, Entropy: 0.0

/Users/sachinsubramanian/Library/Python/3.9/lib/python/site-packages/numpy/lib/_histograms_impl.py:895: RuntimeWarning: divide by zero encountered in divide
  return n/db/n.sum(), bin_edges
/Users/sachinsubramanian/Library/Python/3.9/lib/python/site-packages/numpy/lib/_histograms_impl.py:895: RuntimeWarning: invalid value encountered in divide
  return n/db/n.sum(), bin_edges



vital_wavetable_8 Statistics:
Loudness  → Range: 0.0012, Entropy: 1.0000
Centroid  → Range: 0.0000, Entropy: nan
Flatness  → Range: 0.0000, Entropy: 0.0000
--------------------------------------------------

vital_wavetable_9 Statistics:
Loudness  → Range: 2.5612, Entropy: 1.0000
Centroid  → Range: 0.2126, Entropy: 1.0000
Flatness  → Range: 0.4685, Entropy: 1.0000
--------------------------------------------------

vital_wavetable_10 Statistics:
Loudness  → Range: 2.5612, Entropy: 1.0000
Centroid  → Range: 0.2126, Entropy: 1.0000
Flatness  → Range: 0.4685, Entropy: 1.0000
--------------------------------------------------

vital_wavetable_11 Statistics:
Loudness  → Range: 0.0000, Entropy: 0.0000
Centroid  → Range: 0.0000, Entropy: 0.0000
Flatness  → Range: 0.0000, Entropy: 0.0000
--------------------------------------------------

vital_wavetable_12 Statistics:
Loudness  → Range: 1.3788, Entropy: 1.0000
Centroid  → Range: 0.8728, Entropy: 1.0000
Flatness  → Range: 9.4628, Entropy: 1.0